# Notebook 7 — Encode Geospatial Network into the CANOE Schema

This notebook converts the geospatial outputs developed in previous notebooks into a CANOE-compatible SQLite database.

The objective is to replace the synthetic grid-neighbor representation used in the prototype model with a transport network derived from the Canadian basemap and road connectivity analysis while preserving compatibility with the existing CANOE/TEMOA model structure.

Transport links are represented using CANOE pseudo-regions of the form

```
region_from-region_to
```

where the two regions correspond to adjacent basemap polygons connected by existing infrastructure. This forms a dual graph representation of Canadian infrastructure layers aligned under a specified spatial resolution.

Initially, this notebook focuses on road-based transport technologies and encodes only links for which road connectivity has been identified. The absence of a link implies that transport between those regions is infeasible.

Rather than rebuilding the complete database from raw CSV files, this notebook loads the existing CANOE database and performs a schema reconciliation step. Inherited tables are filtered to the geospatial region topology and augmented with new transport technologies, producing a functional geospatial test database suitable for MILP execution.

The resulting database provides an intermediate development layer between the geospatial preprocessing workflow and eventual integration into the core CANOE modules.

---

## Inputs

### Basemap regions

From Notebook 4:

* Regional polygon geometries
* Region identifiers
* Region centroids

### Neighbor relationships

From Notebook 5:

* Polygon adjacency graph
* Neighbor pairs
* Inter-region distances

### Road connectivity

From Notebook 6:

* Weak road connectivity
* Strong road connectivity

### Existing CANOE database

* Existing CANOE SQLite database
* Schema definitions
* Technology definitions
* Commodity definitions
* Supporting tables

---

## Outputs

This notebook modifies and validates CANOE tables including:

* Region
* Technology
* Efficiency
* CostVariable
* CostInvest
* ETLSegment
* Demand
* LimitCapacity
* Supporting schema tables

and exports complete SQLite databases suitable for direct use by the CANOE/TEMOA solver.

---

## Conceptual workflow

1. Load the regional basemap and road connectivity outputs.
2. Load the existing CANOE database.
3. Replace the synthetic region representation with geospatial regions.
4. Build transport edges from connected neighboring regions.
5. Encode transport technologies for each valid edge.
6. Reconcile inherited node and edge tables with the geospatial topology.
7. Validate schema consistency.
8. Export SQLite databases.
9. Test MILP execution using the existing CANOE workflow.

This notebook serves as a graph-to-schema encoder and schema reconciliation layer between the geospatial preprocessing workflow and eventual integration into the core CANOE modules.

In [1]:
# =============================================================================
# Dependencies and project directories
# =============================================================================

from pathlib import Path
import shutil
import sqlite3

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import db_mgmt


PROJECT_ROOT = Path.cwd().parent

DATA_FILES = PROJECT_ROOT / "data_files"

RAW_BASEMAPS = DATA_FILES / "raw" / "basemaps"

PROCESSED_BASEMAPS = DATA_FILES / "processed" / "basemaps"
PROCESSED_GRAPH = DATA_FILES / "processed" / "graph"
PROCESSED_ROAD_CONNECTIVITY = DATA_FILES / "processed" / "road_connectivity"
PROCESSED_SCHEMA = DATA_FILES / "processed" / "schema"

PROCESSED_SCHEMA.mkdir(
    parents=True,
    exist_ok=True,
)

In [2]:
# =============================================================================
# Select basemap, graph, and road connectivity inputs
# =============================================================================

BASEMAP_STEM = "canada_basemap_1deg_intersects"
CONNECTION_METHOD = "weak"

RAW_BASEMAP_PATH = RAW_BASEMAPS / "lpr_000b21a_e.shp"
RAW_SCHEMA_PATH = DATA_FILES / "canoe_dataset_schema.sql"
BASELINE_SQLITE_PATH = DATA_FILES / "CANOE_geospatial.sqlite"

BASEMAP_PATH = (
    PROCESSED_BASEMAPS
    / f"{BASEMAP_STEM}.gpkg"
)

GRAPH_NODE_PATH = (
    PROCESSED_GRAPH
    / f"{BASEMAP_STEM}_graph_nodes.gpkg"
)

GRAPH_EDGE_PATH = (
    PROCESSED_GRAPH
    / f"{BASEMAP_STEM}_graph_edges.csv"
)

ROAD_EDGE_CONNECTIONS_PATH = (
    PROCESSED_ROAD_CONNECTIVITY
    / f"{BASEMAP_STEM}_road_connectivity_{CONNECTION_METHOD}_road_edge_connections.csv"
)

ROAD_EDGES_GPKG_PATH = (
    PROCESSED_ROAD_CONNECTIVITY
    / f"{BASEMAP_STEM}_road_connectivity_{CONNECTION_METHOD}_road_edges.gpkg"
)

ROAD_REGION_OVERLAY_PATH = (
    PROCESSED_ROAD_CONNECTIVITY
    / f"{BASEMAP_STEM}_road_connectivity_road_region_overlay.gpkg"
)

OUTPUT_SQLITE_PATH = (
    PROCESSED_SCHEMA
    / f"CANOE_geospatial_{BASEMAP_STEM}_roads_{CONNECTION_METHOD}.sqlite"
)


input_paths = {
    "raw_basemap": RAW_BASEMAP_PATH,
    "processed_basemap": BASEMAP_PATH,
    "graph_nodes": GRAPH_NODE_PATH,
    "graph_edges": GRAPH_EDGE_PATH,
    "road_edge_connections": ROAD_EDGE_CONNECTIONS_PATH,
    "road_edges_gpkg": ROAD_EDGES_GPKG_PATH,
    "road_region_overlay": ROAD_REGION_OVERLAY_PATH,
    "raw_schema": RAW_SCHEMA_PATH,
    "baseline_sqlite": BASELINE_SQLITE_PATH,
}

missing_paths = {
    name: path
    for name, path in input_paths.items()
    if not path.exists()
}

if missing_paths:
    for name, path in missing_paths.items():
        print(f"Missing {name}: {path}")
    raise FileNotFoundError("One or more required input files are missing.")

print("Selected configuration:")
print(f"Basemap: {BASEMAP_STEM}")
print(f"Road method: {CONNECTION_METHOD}")

print("\nAll required input files found.")
print(f"Processed basemap: {BASEMAP_PATH.name}")
print(f"Graph nodes: {GRAPH_NODE_PATH.name}")
print(f"Graph edges: {GRAPH_EDGE_PATH.name}")
print(f"Road connections: {ROAD_EDGE_CONNECTIONS_PATH.name}")
print(f"Road edge geometry: {ROAD_EDGES_GPKG_PATH.name}")
print(f"Output SQLite: {OUTPUT_SQLITE_PATH.name}")

Selected configuration:
Basemap: canada_basemap_1deg_intersects
Road method: weak

All required input files found.
Processed basemap: canada_basemap_1deg_intersects.gpkg
Graph nodes: canada_basemap_1deg_intersects_graph_nodes.gpkg
Graph edges: canada_basemap_1deg_intersects_graph_edges.csv
Road connections: canada_basemap_1deg_intersects_road_connectivity_weak_road_edge_connections.csv
Road edge geometry: canada_basemap_1deg_intersects_road_connectivity_weak_road_edges.gpkg
Output SQLite: CANOE_geospatial_canada_basemap_1deg_intersects_roads_weak.sqlite


In [3]:
# =============================================================================
# Load basemap, graph, road, and baseline database inputs
# =============================================================================

basemap = gpd.read_file(
    BASEMAP_PATH,
)

graph_nodes = gpd.read_file(
    GRAPH_NODE_PATH,
)

graph_edges = pd.read_csv(
    GRAPH_EDGE_PATH,
)

road_edge_connections = pd.read_csv(
    ROAD_EDGE_CONNECTIONS_PATH,
)

road_edges_gdf = gpd.read_file(
    ROAD_EDGES_GPKG_PATH,
)

db = db_mgmt.sqlite_to_dfs(
    BASELINE_SQLITE_PATH,
)

print(f"Basemap regions: {len(basemap):,} rows")
print(f"Graph nodes: {len(graph_nodes):,} rows")
print(f"Graph edges: {len(graph_edges):,} rows")
print(f"Road edge connections: {len(road_edge_connections):,} rows")
print(f"Road edge geometries: {len(road_edges_gdf):,} rows")
print(f"Baseline database tables: {len(db):,}")

print("\nCRS:")
print(f"Basemap: {basemap.crs}")
print(f"Graph nodes: {graph_nodes.crs}")
print(f"Road edge geometries: {road_edges_gdf.crs}")

Basemap regions: 2,276 rows
Graph nodes: 2,276 rows
Graph edges: 8,580 rows
Road edge connections: 8,580 rows
Road edge geometries: 1,896 rows
Baseline database tables: 85

CRS:
Basemap: EPSG:4326
Graph nodes: EPSG:4326
Road edge geometries: EPSG:4326


In [4]:
# -----------------------------------------------------------------------------
# Raw input tables from original encoder
# -----------------------------------------------------------------------------

SITES_PATH = DATA_FILES / "sites_full.csv"
DEMAND_PATH = DATA_FILES / "demand.csv"
CO2_PATH = DATA_FILES / "co2.csv"
GEN_EFFICIENCIES_PATH = DATA_FILES / "generation_efficiency.csv"
TECHNOLOGIES_PATH = DATA_FILES / "techs.csv"
COMMODITIES_PATH = DATA_FILES / "commodities.csv"

sites_raw = pd.read_csv(SITES_PATH)
demand_raw = pd.read_csv(DEMAND_PATH)
co2_raw = pd.read_csv(CO2_PATH)
gen_efficiencies_raw = pd.read_csv(GEN_EFFICIENCIES_PATH)
technologies_raw = pd.read_csv(TECHNOLOGIES_PATH)
commodities_raw = pd.read_csv(COMMODITIES_PATH)

In [5]:
# =============================================================================
# Inspect core database and graph inputs
# =============================================================================

core_tables = [
    "Region",
    "Technology",
    "TechnologyType",
    "Commodity",
    "Efficiency",
    "CostVariable",
    "CostInvest",
    "ETLSegment",
    "LimitCapacity",
    "Demand",
    "DataSet",
]

for table in core_tables:
    df = db[table]
    print(f"{table}: {len(df):,} rows")

print("\nGraph input columns:")
print(f"graph_nodes: {list(graph_nodes.columns)}")
print(f"graph_edges: {list(graph_edges.columns)}")
print(f"road_edge_connections: {list(road_edge_connections.columns)}")

display(graph_nodes.head())
display(graph_edges.head())
display(road_edge_connections.head())
display(db["Technology"].sort_values("tech").reset_index(drop=True))
display(db["Commodity"].sort_values("name").reset_index(drop=True))

Region: 2,259 rows
Technology: 12 rows
TechnologyType: 4 rows
Commodity: 7 rows
Efficiency: 62,188 rows
CostVariable: 50,487 rows
CostInvest: 4,518 rows
ETLSegment: 193,552 rows
LimitCapacity: 4,518 rows
Demand: 123 rows
DataSet: 1 rows

Graph input columns:
graph_nodes: ['lon_min', 'lon_max', 'lat_min', 'lat_max', 'lon', 'lat', 'region', 'site_id', 'resolution_deg', 'keep_method', 'up_id', 'down_id', 'right_id', 'left_id', 'n_neighbors', 'up_distance', 'down_distance', 'right_distance', 'left_distance', 'geometry']
graph_edges: ['edge_region', 'region_from', 'region_to', 'direction', 'lon_from', 'lat_from', 'lon_to', 'lat_to', 'distance_km', 'resolution_deg', 'keep_method']
road_edge_connections: ['edge_region', 'region_from', 'region_to', 'direction', 'lon_from', 'lat_from', 'lon_to', 'lat_to', 'distance_km', 'resolution_deg', 'keep_method', 'has_road_from', 'has_road_to', 'has_road_connection', 'connection_method', 'region_pair']


,lon_min,lon_max,lat_min,lat_max,lon,lat,region,site_id,resolution_deg,keep_method,up_id,down_id,right_id,left_id,n_neighbors,up_distance,down_distance,right_distance,left_distance,geometry
0,-84.0,-83.0,41.0,42.0,-83.5,41.5,R0,R0,1.0,intersects,R2,R-999,R1,R-999,2,111.073287,NaN,83.495703,NaN,"POLYGON ((-83 41, -83 42, -84 42, -84 41, -83 ..."
1,-83.0,-82.0,41.0,42.0,-82.5,41.5,R1,R1,1.0,intersects,R3,R-999,R-999,R0,2,111.073287,NaN,NaN,83.495703,"POLYGON ((-82 41, -82 42, -83 42, -83 41, -82 ..."
2,-84.0,-83.0,42.0,43.0,-83.5,42.5,R2,R2,1.0,intersects,R-999,R0,R3,R-999,2,NaN,111.073287,82.198536,NaN,"POLYGON ((-83 42, -83 43, -84 43, -84 42, -83 ..."
3,-83.0,-82.0,42.0,43.0,-82.5,42.5,R3,R3,1.0,intersects,R8,R1,R4,R2,4,111.092738,111.073287,82.198536,82.198536,"POLYGON ((-82 42, -82 43, -83 43, -83 42, -82 ..."
4,-82.0,-81.0,42.0,43.0,-81.5,42.5,R4,R4,1.0,intersects,R9,R-999,R5,R3,3,111.092738,NaN,82.198536,82.198536,"POLYGON ((-81 42, -81 43, -82 43, -82 42, -81 ..."


,edge_region,region_from,region_to,direction,lon_from,lat_from,lon_to,lat_to,distance_km,resolution_deg,keep_method
0,R0-R2,R0,R2,up,-83.5,41.5,-83.5,42.5,111.073287,1.0,intersects
1,R0-R1,R0,R1,right,-83.5,41.5,-82.5,41.5,83.495703,1.0,intersects
2,R1-R3,R1,R3,up,-82.5,41.5,-82.5,42.5,111.073287,1.0,intersects
3,R1-R0,R1,R0,left,-82.5,41.5,-83.5,41.5,83.495703,1.0,intersects
4,R2-R0,R2,R0,down,-83.5,42.5,-83.5,41.5,111.073287,1.0,intersects


,edge_region,region_from,region_to,direction,lon_from,lat_from,lon_to,lat_to,distance_km,resolution_deg,keep_method,has_road_from,has_road_to,has_road_connection,connection_method,region_pair
0,R0-R2,R0,R2,up,-83.5,41.5,-83.5,42.5,111.073287,1.0,intersects,False,True,False,weak_node_presence_adjacency,R0-R2
1,R0-R1,R0,R1,right,-83.5,41.5,-82.5,41.5,83.495703,1.0,intersects,False,True,False,weak_node_presence_adjacency,R0-R1
2,R1-R3,R1,R3,up,-82.5,41.5,-82.5,42.5,111.073287,1.0,intersects,True,True,True,weak_node_presence_adjacency,R1-R3
3,R1-R0,R1,R0,left,-82.5,41.5,-83.5,41.5,83.495703,1.0,intersects,True,False,False,weak_node_presence_adjacency,R1-R0
4,R2-R0,R2,R0,down,-83.5,42.5,-83.5,41.5,111.073287,1.0,intersects,True,False,False,weak_node_presence_adjacency,R2-R0


,tech,flag,sector,category,sub_category,unlim_cap,annual,reserve,curtail,retire,flex,exchange,seas_stor,description,data_id
0,CO2_CAP,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
1,CO2_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
2,ELC_GEN,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
3,ELC_TRANS,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
4,GSL_BACKUP,p,industrial,None,None,1,1,0,0,0,0,0,0,None,GEO001
5,GSL_DEMAND,p,industrial,None,None,1,1,0,0,0,0,0,0,None,GEO001
6,GSL_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
7,GSL_PLANT,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
8,H2_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
9,H2_PLANT,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001


,name,flag,description,data_id
0,ch3oh,wa,methanol,None
1,co2,wa,co2 captured,None
2,d_gsl,d,gasoline demand,None
3,elc,wa,electricity,None
4,ethos,s,dummy,None
5,gsl,wa,gasoline,None
6,h2,wa,hydrogen,None


In [6]:
# =============================================================================
# Build canonical region and transport-link tables
# =============================================================================

region_table = (
    graph_nodes[["region"]]
    .drop_duplicates()
    .sort_values(
        "region",
        key=lambda s: s.str.extract(r"R(\d+)")[0].astype(int),
    )
    .reset_index(drop=True)
)

region_table["notes"] = (
    f"{BASEMAP_STEM} CANOE geospatial graph node"
)

road_links = (
    road_edge_connections
    .loc[road_edge_connections["has_road_connection"]]
    .copy()
)

road_links = road_links[
    [
        "edge_region",
        "region_from",
        "region_to",
        "direction",
        "connection_method",
        "distance_km",
        "lon_from",
        "lat_from",
        "lon_to",
        "lat_to",
    ]
].copy()

road_links = (
    road_links
    .drop_duplicates(subset=["edge_region"])
    .reset_index(drop=True)
)

road_links["canoe_region"] = road_links["edge_region"]

pipeline_links = graph_edges.copy()
pipeline_links["canoe_region"] = pipeline_links["edge_region"]

VALID_NODE_REGIONS = set(region_table["region"])
VALID_PIPELINE_EDGE_REGIONS = set(pipeline_links["canoe_region"])
VALID_ROAD_EDGE_REGIONS = set(road_links["canoe_region"])

print(f"Region table rows: {len(region_table):,}")
print(f"Pipeline links: {len(pipeline_links):,}")
print(f"Road links ({CONNECTION_METHOD}): {len(road_links):,}")

display(region_table.head())
display(pipeline_links.head())
display(road_links.head())

Region table rows: 2,276
Pipeline links: 8,580
Road links (weak): 1,896


,region,notes
0,R0,canada_basemap_1deg_intersects CANOE geospatia...
1,R1,canada_basemap_1deg_intersects CANOE geospatia...
2,R2,canada_basemap_1deg_intersects CANOE geospatia...
3,R3,canada_basemap_1deg_intersects CANOE geospatia...
4,R4,canada_basemap_1deg_intersects CANOE geospatia...


,edge_region,region_from,region_to,direction,lon_from,lat_from,lon_to,lat_to,distance_km,resolution_deg,keep_method,canoe_region
0,R0-R2,R0,R2,up,-83.5,41.5,-83.5,42.5,111.073287,1.0,intersects,R0-R2
1,R0-R1,R0,R1,right,-83.5,41.5,-82.5,41.5,83.495703,1.0,intersects,R0-R1
2,R1-R3,R1,R3,up,-82.5,41.5,-82.5,42.5,111.073287,1.0,intersects,R1-R3
3,R1-R0,R1,R0,left,-82.5,41.5,-83.5,41.5,83.495703,1.0,intersects,R1-R0
4,R2-R0,R2,R0,down,-83.5,42.5,-83.5,41.5,111.073287,1.0,intersects,R2-R0


,edge_region,region_from,region_to,direction,connection_method,distance_km,lon_from,lat_from,lon_to,lat_to,canoe_region
0,R1-R3,R1,R3,up,weak_node_presence_adjacency,111.073287,-82.5,41.5,-82.5,42.5,R1-R3
1,R2-R3,R2,R3,right,weak_node_presence_adjacency,82.198536,-83.5,42.5,-82.5,42.5,R2-R3
2,R3-R8,R3,R8,up,weak_node_presence_adjacency,111.092738,-82.5,42.5,-82.5,43.5,R3-R8
3,R3-R1,R3,R1,down,weak_node_presence_adjacency,111.073287,-82.5,42.5,-82.5,41.5,R3-R1
4,R3-R4,R3,R4,right,weak_node_presence_adjacency,82.198536,-82.5,42.5,-81.5,42.5,R3-R4


In [7]:
# =============================================================================
# Validate canonical region and transport-link tables
# =============================================================================

assert "R-999" not in set(region_table["region"])
assert len(region_table) == region_table["region"].nunique()
assert len(region_table) == len(graph_nodes)

assert pipeline_links["canoe_region"].nunique() == len(pipeline_links)
assert road_links["canoe_region"].nunique() == len(road_links)

assert set(pipeline_links["region_from"]).issubset(VALID_NODE_REGIONS)
assert set(pipeline_links["region_to"]).issubset(VALID_NODE_REGIONS)

assert set(road_links["region_from"]).issubset(VALID_NODE_REGIONS)
assert set(road_links["region_to"]).issubset(VALID_NODE_REGIONS)

assert pipeline_links["distance_km"].notna().all()
assert road_links["distance_km"].notna().all()

assert (pipeline_links["distance_km"] > 0).all()
assert (road_links["distance_km"] > 0).all()

assert pipeline_links["canoe_region"].str.contains("-", regex=False).all()
assert road_links["canoe_region"].str.contains("-", regex=False).all()

print("Canonical region and transport-link tables validated.")

print("\nDistance summaries:")
display(
    pd.DataFrame(
        {
            "pipeline_km": pipeline_links["distance_km"].describe(),
            "road_km": road_links["distance_km"].describe(),
        }
    )
)

Canonical region and transport-link tables validated.

Distance summaries:


,pipeline_km,road_km
count,8580.000000,1896.000000
mean,80.389393,88.060567
std,32.816282,22.397159
min,12.643413,39.099465
25%,51.537011,69.439977
50%,78.157559,78.157559
75%,111.445558,111.248271
max,111.677184,111.549102


In [8]:
# =============================================================================
# Initialize encoded database and define transport technologies
# =============================================================================

db_encoded = {
    table_name: df.copy()
    for table_name, df in db.items()
}

db_encoded["Region"] = region_table.copy()


truck_tech_specs = pd.DataFrame(
    [
        {
            "tech": "CO2_TRUCK",
            "input_comm": "co2",
            "output_comm": "co2",
            "description": "Road transport of carbon dioxide by truck",
        },
        {
            "tech": "H2_TRUCK",
            "input_comm": "h2",
            "output_comm": "h2",
            "description": "Road transport of hydrogen by truck",
        },
        {
            "tech": "GSL_TRUCK",
            "input_comm": "gsl",
            "output_comm": "gsl",
            "description": "Road transport of gasoline by truck",
        },
        {
            "tech": "METOH_TRUCK",
            "input_comm": "ch3oh",
            "output_comm": "ch3oh",
            "description": "Road transport of methanol by truck",
        },
    ]
)

pipeline_tech_specs = pd.DataFrame(
    [
        {"tech": "CO2_PIPE", "input_comm": "co2", "output_comm": "co2"},
        {"tech": "H2_PIPE", "input_comm": "h2", "output_comm": "h2"},
        {"tech": "GSL_PIPE", "input_comm": "gsl", "output_comm": "gsl"},
        {"tech": "METOH_PIPE", "input_comm": "ch3oh", "output_comm": "ch3oh"},
    ]
)

PIPE_TECHS = set(pipeline_tech_specs["tech"])
TRUCK_TECHS = set(truck_tech_specs["tech"])

print(f"Baseline Region rows: {len(db['Region']):,}")
print(f"Encoded Region rows: {len(db_encoded['Region']):,}")

display(truck_tech_specs)
display(pipeline_tech_specs)

Baseline Region rows: 2,259
Encoded Region rows: 2,276


,tech,input_comm,output_comm,description
0,CO2_TRUCK,co2,co2,Road transport of carbon dioxide by truck
1,H2_TRUCK,h2,h2,Road transport of hydrogen by truck
2,GSL_TRUCK,gsl,gsl,Road transport of gasoline by truck
3,METOH_TRUCK,ch3oh,ch3oh,Road transport of methanol by truck


,tech,input_comm,output_comm
0,CO2_PIPE,co2,co2
1,H2_PIPE,h2,h2
2,GSL_PIPE,gsl,gsl
3,METOH_PIPE,ch3oh,ch3oh


In [9]:
# =============================================================================
# Snap point inputs to selected graph-node polygons
# =============================================================================

def snap_points_to_graph_nodes(
    points: pd.DataFrame,
    graph_nodes: gpd.GeoDataFrame,
    lon_col: str = "lon",
    lat_col: str = "lat",
) -> pd.DataFrame:

    points_gdf = gpd.GeoDataFrame(
        points.copy(),
        geometry=gpd.points_from_xy(points[lon_col], points[lat_col]),
        crs="EPSG:4326",
    )

    nodes_gdf = graph_nodes[
        [
            "region",
            "site_id",
            "lon",
            "lat",
            "geometry",
        ]
    ].copy()

    snapped = gpd.sjoin_nearest(
        points_gdf.to_crs("EPSG:3347"),
        nodes_gdf.to_crs("EPSG:3347"),
        how="left",
        distance_col="snap_distance_m",
    )

    snapped = snapped.to_crs("EPSG:4326")

    return pd.DataFrame(
        snapped.drop(columns="geometry")
    )


raw_points = pd.concat(
    [
        sites_raw,
        demand_raw,
        co2_raw,
    ],
    ignore_index=True,
).fillna(0)

snapped_points = snap_points_to_graph_nodes(
    raw_points,
    graph_nodes,
)

site_attributes = (
    snapped_points
    .groupby("region", as_index=False)
    .agg(
        LCOE=("LCOE", "mean"),
        max_elc=("max_elec", "sum"),
        demand=("demand", "sum"),
        co2=("CO2", "sum"),
        max_snap_distance_m=("snap_distance_m", "max"),
    )
)

site_attributes = (
    region_table[["region"]]
    .merge(site_attributes, on="region", how="left")
    .fillna(
        {
            "LCOE": 0,
            "max_elc": 0,
            "demand": 0,
            "co2": 0,
            "max_snap_distance_m": 0,
        }
    )
)

site_attributes["co2_cost"] = 50

print(f"Snapped site attribute rows: {len(site_attributes):,}")
print(f"Regions with demand: {(site_attributes['demand'] > 0).sum():,}")
print(f"Regions with CO2: {(site_attributes['co2'] > 0).sum():,}")
print(f"Regions with electricity potential: {(site_attributes['max_elc'] > 0).sum():,}")

display(site_attributes.head())
display(site_attributes["max_snap_distance_m"].describe())

Snapped site attribute rows: 2,276
Regions with demand: 123
Regions with CO2: 63
Regions with electricity potential: 2,203


,region,LCOE,max_elc,demand,co2,max_snap_distance_m,co2_cost
0,R0,0.000000,0.000000e+00,0.0,0.0,0.0,50
1,R1,0.026981,2.143874e+06,0.0,0.0,0.0,50
2,R2,0.027066,9.966600e+05,0.0,24990.0,0.0,50
3,R3,0.023377,7.369721e+06,0.0,2189710.0,0.0,50
4,R4,0.030312,5.761010e+06,0.0,0.0,0.0,50


count     2276.000000
mean        68.173342
std        771.673592
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max      19230.308821
Name: max_snap_distance_m, dtype: float64

In [10]:
# =============================================================================
# Replace node-level Demand and LimitCapacity from snapped attributes
# =============================================================================

db_encoded["Demand"] = pd.DataFrame(
    {
        "region": site_attributes.loc[
            site_attributes["demand"] > 0,
            "region",
        ],
        "period": 1,
        "commodity": "d_gsl",
        "demand": site_attributes.loc[
            site_attributes["demand"] > 0,
            "demand",
        ],
        "units": None,
        "notes": "Demand snapped to selected geospatial graph node",
        "data_source": None,
        "dq_cred": None,
        "dq_geog": None,
        "dq_struc": None,
        "dq_tech": None,
        "dq_time": None,
        "data_id": "GEO001",
    }
)

db_encoded["LimitCapacity"] = pd.concat(
    [
        pd.DataFrame(
            {
                "region": site_attributes["region"],
                "period": 1,
                "tech_or_group": "CO2_CAP",
                "operator": "le",
                "capacity": site_attributes["co2"],
                "units": None,
                "notes": "CO2 capacity snapped to selected geospatial graph node",
                "data_source": None,
                "dq_cred": None,
                "dq_geog": None,
                "dq_struc": None,
                "dq_tech": None,
                "dq_time": None,
                "data_id": "GEO001",
            }
        ),
        pd.DataFrame(
            {
                "region": site_attributes["region"],
                "period": 1,
                "tech_or_group": "ELC_GEN",
                "operator": "le",
                "capacity": site_attributes["max_elc"],
                "units": None,
                "notes": "Electricity potential snapped to selected geospatial graph node",
                "data_source": None,
                "dq_cred": None,
                "dq_geog": None,
                "dq_struc": None,
                "dq_tech": None,
                "dq_time": None,
                "data_id": "GEO001",
            }
        ),
    ],
    ignore_index=True,
)

print(f"Demand rows: {len(db_encoded['Demand']):,}")
print(f"LimitCapacity rows: {len(db_encoded['LimitCapacity']):,}")

Demand rows: 123
LimitCapacity rows: 4,552


In [11]:
# =============================================================================
# Rebuild node-level CostVariable and CostInvest rows
# =============================================================================

NODE_COSTVARIABLE_TECHS = [
    "ELC_GEN",
    "CO2_CAP",
    "GSL_BACKUP",
]

node_costvariable = pd.concat(
    [
        pd.DataFrame(
            {
                "region": site_attributes["region"],
                "period": 1,
                "tech": "ELC_GEN",
                "vintage": 1,
                "cost": site_attributes["LCOE"],
                "units": "M$/MWh",
                "notes": "Electricity generation cost snapped to selected graph node",
                "data_source": None,
                "dq_cred": None,
                "dq_geog": None,
                "dq_struc": None,
                "dq_tech": None,
                "dq_time": None,
                "data_id": "GEO001",
            }
        ),
        pd.DataFrame(
            {
                "region": site_attributes["region"],
                "period": 1,
                "tech": "CO2_CAP",
                "vintage": 1,
                "cost": site_attributes["co2_cost"],
                "units": "M$/t",
                "notes": "CO2 capture cost snapped to selected graph node",
                "data_source": None,
                "dq_cred": None,
                "dq_geog": None,
                "dq_struc": None,
                "dq_tech": None,
                "dq_time": None,
                "data_id": "GEO001",
            }
        ),
        pd.DataFrame(
            {
                "region": site_attributes["region"],
                "period": 1,
                "tech": "GSL_BACKUP",
                "vintage": 1,
                "cost": 500000,
                "units": "M$/MWh",
                "notes": "Backup gasoline supply cost",
                "data_source": None,
                "dq_cred": None,
                "dq_geog": None,
                "dq_struc": None,
                "dq_tech": None,
                "dq_time": None,
                "data_id": "GEO001",
            }
        ),
    ],
    ignore_index=True,
)

db_encoded["CostVariable"] = (
    db_encoded["CostVariable"]
    .loc[
        ~db_encoded["CostVariable"]["tech"].isin(NODE_COSTVARIABLE_TECHS)
    ]
    .copy()
)

db_encoded["CostVariable"] = pd.concat(
    [
        db_encoded["CostVariable"],
        node_costvariable,
    ],
    ignore_index=True,
)


NODE_COSTINVEST_TECHS = [
    "ELC_GEN",
    "CO2_CAP",
]

node_costinvest = pd.concat(
    [
        pd.DataFrame(
            {
                "region": site_attributes["region"],
                "tech": "ELC_GEN",
                "vintage": 1,
                "cost": 1000,
                "units": None,
                "notes": "Electricity generation fixed investment cost",
                "data_source": None,
                "dq_cred": None,
                "dq_geog": None,
                "dq_struc": None,
                "dq_tech": None,
                "dq_time": None,
                "data_id": "GEO001",
            }
        ),
        pd.DataFrame(
            {
                "region": site_attributes["region"],
                "tech": "CO2_CAP",
                "vintage": 1,
                "cost": 1000,
                "units": None,
                "notes": "CO2 capture fixed investment cost",
                "data_source": None,
                "dq_cred": None,
                "dq_geog": None,
                "dq_struc": None,
                "dq_tech": None,
                "dq_time": None,
                "data_id": "GEO001",
            }
        ),
    ],
    ignore_index=True,
)

db_encoded["CostInvest"] = (
    db_encoded["CostInvest"]
    .loc[
        ~db_encoded["CostInvest"]["tech"].isin(NODE_COSTINVEST_TECHS)
    ]
    .copy()
)

db_encoded["CostInvest"] = pd.concat(
    [
        db_encoded["CostInvest"],
        node_costinvest,
    ],
    ignore_index=True,
)

print(f"Node CostVariable rows: {len(node_costvariable):,}")
print(f"Node CostInvest rows: {len(node_costinvest):,}")

Node CostVariable rows: 6,828
Node CostInvest rows: 4,552


In [12]:
# =============================================================================
# Rebuild LimitTechInputSplitAnnual rows
# =============================================================================

def build_input_split(
    regions: pd.Series,
    tech: str,
    input_comm: list[str],
    proportion: list[float],
    operator: str = "ge",
) -> pd.DataFrame:

    if len(input_comm) != len(proportion):
        raise ValueError("input_comm and proportion must have same length.")

    rows = []

    for comm, prop in zip(input_comm, proportion):

        rows.append(
            pd.DataFrame(
                {
                    "region": regions,
                    "period": 1,
                    "input_comm": comm,
                    "tech": tech,
                    "operator": operator,
                    "proportion": prop,
                    "notes": None,
                    "data_source": None,
                    "dq_cred": None,
                    "dq_geog": None,
                    "dq_struc": None,
                    "dq_tech": None,
                    "dq_time": None,
                    "data_id": "GEO001",
                }
            )
        )

    return pd.concat(rows, ignore_index=True)


node_regions = site_attributes["region"]

gsl_input_split = build_input_split(
    node_regions,
    "GSL_PLANT",
    ["ch3oh", "h2"],
    [0.997782705, 0.002217295],
)

metoh_input_split = build_input_split(
    node_regions,
    "METOH_PLANT",
    ["co2", "h2", "elc"],
    [0.79230333899, 0.10865874363, 0.09903791737],
)

db_encoded["LimitTechInputSplitAnnual"] = pd.concat(
    [
        gsl_input_split,
        metoh_input_split,
    ],
    ignore_index=True,
)

print(
    f"LimitTechInputSplitAnnual rows: "
    f"{len(db_encoded['LimitTechInputSplitAnnual']):,}"
)

LimitTechInputSplitAnnual rows: 11,380


In [13]:
# =============================================================================
# Rebuild node-level production and demand Efficiency rows
# =============================================================================

efficiency_rows = []

for row in gen_efficiencies_raw.itertuples(index=False):

    if row.tech == "GSL_BACKUP":
        regions = site_attributes.loc[
            site_attributes["demand"] > 0,
            "region",
        ]
    else:
        regions = site_attributes["region"]

    efficiency_rows.append(
        pd.DataFrame(
            {
                "region": regions,
                "input_comm": row.input_comm,
                "tech": row.tech,
                "vintage": 1,
                "output_comm": row.output_comm,
                "efficiency": row.efficiency,
                "notes": "Node-level efficiency rebuilt from snapped graph regions",
                "data_source": None,
                "dq_cred": None,
                "dq_geog": None,
                "dq_struc": None,
                "dq_tech": None,
                "dq_time": None,
                "data_id": "GEO001",
            }
        )
    )

node_efficiency = pd.concat(
    efficiency_rows,
    ignore_index=True,
)

demand_regions = (
    db_encoded["Demand"]["region"]
    .drop_duplicates()
    .reset_index(drop=True)
)

gsl_demand_efficiency = pd.DataFrame(
    {
        "region": demand_regions,
        "input_comm": "gsl",
        "tech": "GSL_DEMAND",
        "vintage": 1,
        "output_comm": "d_gsl",
        "efficiency": 1.0,
        "notes": "Gasoline demand technology rebuilt from snapped demand regions",
        "data_source": None,
        "dq_cred": None,
        "dq_geog": None,
        "dq_struc": None,
        "dq_tech": None,
        "dq_time": None,
        "data_id": "GEO001",
    }
)

node_efficiency = pd.concat(
    [
        node_efficiency,
        gsl_demand_efficiency,
    ],
    ignore_index=True,
)

NODE_EFFICIENCY_TECHS = set(node_efficiency["tech"])

# Remove inherited rows for node-level technologies and all inherited edge rows.
# Edge rows are rebuilt later from the selected graph/road connectivity.
db_encoded["Efficiency"] = (
    db_encoded["Efficiency"]
    .loc[
        (
            ~db_encoded["Efficiency"]["tech"].isin(NODE_EFFICIENCY_TECHS)
        )
        &
        (
            ~db_encoded["Efficiency"]["region"]
            .astype(str)
            .str.contains("-", regex=False)
        )
    ]
    .copy()
)

db_encoded["Efficiency"] = pd.concat(
    [
        db_encoded["Efficiency"],
        node_efficiency,
    ],
    ignore_index=True,
)

assert set(db_encoded["Demand"]["region"]) == set(
    db_encoded["Efficiency"]
    .loc[
        db_encoded["Efficiency"]["tech"] == "GSL_DEMAND",
        "region",
    ]
)

assert not (
    db_encoded["Efficiency"]["region"]
    .astype(str)
    .str.contains("-", regex=False)
).any(), (
    "Edge-region Efficiency rows should not exist yet. "
    "Transport Efficiency rows must be rebuilt later from selected graph links."
)

print(f"Node Efficiency rows: {len(node_efficiency):,}")
print(f"Encoded Efficiency rows after node rebuild: {len(db_encoded['Efficiency']):,}")

Node Efficiency rows: 18,454
Encoded Efficiency rows after node rebuild: 18,454


In [14]:
# =============================================================================
# Rebuild ETLSegment rows for plants and pipelines
# =============================================================================

PLANT_TECHS = {
    "GSL_PLANT",
    "METOH_PLANT",
}

PIPE_TECHS = set(
    pipeline_tech_specs["tech"]
)


# -----------------------------------------------------------------------------
# Build plant ETLSegment rows for selected node regions
# -----------------------------------------------------------------------------

plant_etl_template = (
    db["ETLSegment"]
    .loc[
        db["ETLSegment"]["tech_or_group"].isin(PLANT_TECHS)
    ]
    .copy()
)

assert not plant_etl_template.empty, (
    "No plant ETLSegment rows found in baseline database."
)

plant_segments = (
    plant_etl_template
    .drop(columns=["region"])
    .drop_duplicates()
    .sort_values(["tech_or_group", "segment"])
    .reset_index(drop=True)
)

plant_etl_rows = []

for tech in sorted(PLANT_TECHS):

    tech_segments = (
        plant_segments
        .loc[
            plant_segments["tech_or_group"] == tech
        ]
        .copy()
    )

    assert not tech_segments.empty, (
        f"No ETLSegment template rows found for {tech}."
    )

    region_frame = pd.DataFrame(
        {
            "region": site_attributes["region"],
            "key": 1,
        }
    )

    segment_frame = tech_segments.copy()
    segment_frame["key"] = 1

    df = (
        region_frame
        .merge(
            segment_frame,
            on="key",
        )
        .drop(columns="key")
    )

    plant_etl_rows.append(df)

plant_etl_new = pd.concat(
    plant_etl_rows,
    ignore_index=True,
)


# -----------------------------------------------------------------------------
# Build pipeline ETLSegment rows for selected graph edge regions
# -----------------------------------------------------------------------------

pipeline_etl_template = (
    db["ETLSegment"]
    .loc[
        db["ETLSegment"]["tech_or_group"] == "H2_PIPE"
    ]
    .copy()
)

assert not pipeline_etl_template.empty, (
    "No H2_PIPE ETLSegment rows found in baseline database."
)

assert pipeline_etl_template["segment"].nunique() > 1, (
    "H2_PIPE ETLSegment template has only one segment. "
    "This would collapse the piecewise investment formulation."
)

pipeline_segments = (
    pipeline_etl_template
    .drop(columns=["region", "tech_or_group"])
    .drop_duplicates()
    .sort_values("segment")
    .reset_index(drop=True)
)

pipeline_etl_rows = []

for pipe in pipeline_tech_specs.itertuples(index=False):

    edge_frame = pd.DataFrame(
        {
            "region": pipeline_links["canoe_region"],
            "key": 1,
        }
    )

    segment_frame = pipeline_segments.copy()
    segment_frame["key"] = 1

    df = (
        edge_frame
        .merge(
            segment_frame,
            on="key",
        )
        .drop(columns="key")
    )

    df["tech_or_group"] = pipe.tech

    df = df[
        [
            "region",
            "tech_or_group",
            "segment",
            "cap_lower",
            "cap_upper",
            "cost_lower",
            "cost_upper",
            "data_id",
        ]
    ].copy()

    pipeline_etl_rows.append(df)

pipeline_etl_new = pd.concat(
    pipeline_etl_rows,
    ignore_index=True,
)


# -----------------------------------------------------------------------------
# Replace ETLSegment geography
# -----------------------------------------------------------------------------

# Keep inherited non-edge ETLSegment rows except plant technologies rebuilt above.
# Drop all inherited edge ETLSegment rows because those edge regions belong to
# the old graph, not the selected graph.
non_edge_etl = (
    db_encoded["ETLSegment"]
    .loc[
        ~db_encoded["ETLSegment"]["region"]
        .astype(str)
        .str.contains("-", regex=False)
    ]
    .copy()
)

non_rebuilt_node_etl = (
    non_edge_etl
    .loc[
        ~non_edge_etl["tech_or_group"].isin(PLANT_TECHS)
    ]
    .copy()
)

db_encoded["ETLSegment"] = pd.concat(
    [
        non_rebuilt_node_etl,
        plant_etl_new,
        pipeline_etl_new,
    ],
    ignore_index=True,
)


# -----------------------------------------------------------------------------
# Validate ETLSegment rebuild
# -----------------------------------------------------------------------------

assert (
    db_encoded["ETLSegment"][
        [
            "region",
            "tech_or_group",
            "segment",
        ]
    ]
    .duplicated()
    .sum()
    == 0
), "Duplicate ETLSegment primary keys found."

etl_edge_regions = set(
    db_encoded["ETLSegment"]
    .loc[
        db_encoded["ETLSegment"]["region"]
        .astype(str)
        .str.contains("-", regex=False),
        "region",
    ]
)

invalid_etl_edge_regions = sorted(
    etl_edge_regions
    - VALID_PIPELINE_EDGE_REGIONS
)

assert not invalid_etl_edge_regions, (
    "ETLSegment contains invalid edge regions: "
    f"{invalid_etl_edge_regions[:10]}"
)

print(f"Plant ETLSegment rows: {len(plant_etl_new):,}")
print(f"Pipeline ETLSegment rows: {len(pipeline_etl_new):,}")
print(f"Encoded ETLSegment rows: {len(db_encoded['ETLSegment']):,}")

display(
    db_encoded["ETLSegment"]
    .groupby("tech_or_group")["segment"]
    .nunique()
    .sort_index()
)

Plant ETLSegment rows: 18,208
Pipeline ETLSegment rows: 137,280
Encoded ETLSegment rows: 155,488


tech_or_group
CO2_PIPE       4
GSL_PIPE       4
GSL_PLANT      4
H2_PIPE        4
METOH_PIPE     4
METOH_PLANT    4
Name: segment, dtype: int64

In [15]:
# =============================================================================
# Add truck technologies
# =============================================================================

technology_template = (
    db_encoded["Technology"]
    .loc[db_encoded["Technology"]["tech"] == "H2_PIPE"]
    .copy()
)

assert len(technology_template) == 1

truck_technology = pd.concat(
    [
        technology_template.assign(
            tech=row.tech,
            description=row.description,
        )
        for row in truck_tech_specs.itertuples(index=False)
    ],
    ignore_index=True,
)

db_encoded["Technology"] = (
    pd.concat(
        [db_encoded["Technology"], truck_technology],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["tech", "data_id"],
        keep="last",
    )
)

assert TRUCK_TECHS.issubset(set(db_encoded["Technology"]["tech"]))

print(f"Technology rows: {len(db_encoded['Technology']):,}")
display(db_encoded["Technology"].sort_values("tech"))

Technology rows: 16


,tech,flag,sector,category,sub_category,unlim_cap,annual,reserve,curtail,retire,flex,exchange,seas_stor,description,data_id
2,CO2_CAP,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
7,CO2_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
12,CO2_TRUCK,p,industrial,None,None,0,1,0,0,0,0,1,0,Road transport of carbon dioxide by truck,GEO001
0,ELC_GEN,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
5,ELC_TRANS,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
11,GSL_BACKUP,p,industrial,None,None,1,1,0,0,0,0,0,0,None,GEO001
10,GSL_DEMAND,p,industrial,None,None,1,1,0,0,0,0,0,0,None,GEO001
9,GSL_PIPE,p,industrial,None,None,0,1,0,0,0,0,1,0,None,GEO001
4,GSL_PLANT,p,industrial,None,None,0,1,0,0,0,0,0,0,None,GEO001
14,GSL_TRUCK,p,industrial,None,None,0,1,0,0,0,0,1,0,Road transport of gasoline by truck,GEO001


In [16]:
# =============================================================================
# Build transport Efficiency rows
# =============================================================================

def build_transport_efficiency(
    links: pd.DataFrame,
    tech_specs: pd.DataFrame,
    notes: str,
) -> pd.DataFrame:

    rows = []

    for tech in tech_specs.itertuples(index=False):

        df = pd.DataFrame(
            {
                "region": links["canoe_region"],
                "input_comm": tech.input_comm,
                "tech": tech.tech,
                "vintage": 1,
                "output_comm": tech.output_comm,
                "efficiency": 1.0,
                "notes": notes,
                "data_source": None,
                "dq_cred": None,
                "dq_geog": None,
                "dq_struc": None,
                "dq_tech": None,
                "dq_time": None,
                "data_id": "GEO001",
            }
        )

        rows.append(df)

    return pd.concat(rows, ignore_index=True)


pipeline_efficiency = build_transport_efficiency(
    pipeline_links,
    pipeline_tech_specs,
    "Candidate pipeline transport link on canonical graph edge",
)

truck_efficiency = build_transport_efficiency(
    road_links,
    truck_tech_specs,
    f"Existing {CONNECTION_METHOD} road-connected transport link",
)

TRANSPORT_TECHS = PIPE_TECHS | TRUCK_TECHS

db_encoded["Efficiency"] = (
    db_encoded["Efficiency"]
    .loc[
        ~db_encoded["Efficiency"]["tech"].isin(TRANSPORT_TECHS)
    ]
    .copy()
)

db_encoded["Efficiency"] = pd.concat(
    [
        db_encoded["Efficiency"],
        pipeline_efficiency,
        truck_efficiency,
    ],
    ignore_index=True,
)

assert set(pipeline_efficiency["region"]) == VALID_PIPELINE_EDGE_REGIONS
assert set(truck_efficiency["region"]) == VALID_ROAD_EDGE_REGIONS

print(f"Pipeline Efficiency rows: {len(pipeline_efficiency):,}")
print(f"Truck Efficiency rows: {len(truck_efficiency):,}")
print(f"Encoded Efficiency rows: {len(db_encoded['Efficiency']):,}")

Pipeline Efficiency rows: 34,320
Truck Efficiency rows: 7,584
Encoded Efficiency rows: 60,358


In [17]:
# =============================================================================
# Build transport CostVariable rows
# =============================================================================

PLACEHOLDER_TRUCK_COST_PER_KM = 0.01
PLACEHOLDER_TRUCK_INTERCEPT_COST = 0.0

PLACEHOLDER_PIPE_COST_PER_KM = 0.01
PLACEHOLDER_PIPE_INTERCEPT_COST = 0.0


def build_transport_costvariable(
    links: pd.DataFrame,
    tech_specs: pd.DataFrame,
    cost_per_km: float,
    intercept_cost: float,
    notes: str,
) -> pd.DataFrame:

    rows = []

    assert links["distance_km"].notna().all()
    assert (links["distance_km"] > 0).all()
    assert links["canoe_region"].nunique() == len(links)

    for tech in tech_specs.itertuples(index=False):

        df = pd.DataFrame(
            {
                "region": links["canoe_region"],
                "period": 1,
                "tech": tech.tech,
                "vintage": 1,
                "cost": (
                    intercept_cost
                    + cost_per_km * links["distance_km"]
                ),
                "units": "M$/unit",
                "notes": notes,
                "data_source": None,
                "dq_cred": None,
                "dq_geog": None,
                "dq_struc": None,
                "dq_tech": None,
                "dq_time": None,
                "data_id": "GEO001",
            }
        )

        rows.append(df)

    out = pd.concat(rows, ignore_index=True)

    assert (
        out[
            ["region", "period", "tech", "vintage", "data_id"]
        ].duplicated().sum() == 0
    )

    assert out["cost"].notna().all()
    assert (out["cost"] >= 0).all()

    return out


pipeline_costvariable = build_transport_costvariable(
    pipeline_links,
    pipeline_tech_specs,
    PLACEHOLDER_PIPE_COST_PER_KM,
    PLACEHOLDER_PIPE_INTERCEPT_COST,
    "Placeholder pipeline transport cost based on graph-edge centroid distance",
)

truck_costvariable = build_transport_costvariable(
    road_links,
    truck_tech_specs,
    PLACEHOLDER_TRUCK_COST_PER_KM,
    PLACEHOLDER_TRUCK_INTERCEPT_COST,
    f"Placeholder truck transport cost based on {CONNECTION_METHOD} road-connected graph distance",
)

# Remove all inherited edge-region CostVariable rows.
# These are old baseline transport links and are not valid for the selected graph.
non_edge_costvariable = db_encoded["CostVariable"].loc[
    ~db_encoded["CostVariable"]["region"]
    .astype(str)
    .str.contains("-", regex=False)
].copy()

db_encoded["CostVariable"] = pd.concat(
    [
        non_edge_costvariable,
        pipeline_costvariable,
        truck_costvariable,
    ],
    ignore_index=True,
)

assert set(pipeline_costvariable["region"]) == VALID_PIPELINE_EDGE_REGIONS
assert set(truck_costvariable["region"]) == VALID_ROAD_EDGE_REGIONS

print(f"Pipeline CostVariable rows: {len(pipeline_costvariable):,}")
print(f"Truck CostVariable rows: {len(truck_costvariable):,}")
print(f"Encoded CostVariable rows: {len(db_encoded['CostVariable']):,}")

Pipeline CostVariable rows: 34,320
Truck CostVariable rows: 7,584
Encoded CostVariable rows: 48,732


In [18]:
# =============================================================================
# Build and add zero truck CostInvest rows
# =============================================================================

truck_costinvest_rows = []

for truck in truck_tech_specs.itertuples(index=False):

    df = pd.DataFrame(
        {
            "region": road_links["canoe_region"],
            "tech": truck.tech,
            "vintage": 1,
            "cost": 0.0,
            "units": "M$/unit",
            "notes": "Existing road transport link; no road construction investment encoded",
            "data_source": None,
            "dq_cred": None,
            "dq_geog": None,
            "dq_struc": None,
            "dq_tech": None,
            "dq_time": None,
            "data_id": "GEO001",
        }
    )

    truck_costinvest_rows.append(df)

truck_costinvest = pd.concat(
    truck_costinvest_rows,
    ignore_index=True,
)

db_encoded["CostInvest"] = (
    db_encoded["CostInvest"]
    .loc[
        ~db_encoded["CostInvest"]["tech"].isin(TRUCK_TECHS)
    ]
    .copy()
)

db_encoded["CostInvest"] = pd.concat(
    [
        db_encoded["CostInvest"],
        truck_costinvest,
    ],
    ignore_index=True,
)

assert set(truck_costinvest["region"]) == VALID_ROAD_EDGE_REGIONS

print(f"Truck CostInvest rows: {len(truck_costinvest):,}")
print(f"Encoded CostInvest rows: {len(db_encoded['CostInvest']):,}")

Truck CostInvest rows: 7,584
Encoded CostInvest rows: 12,136


In [19]:
# =============================================================================
# Validate encoded transport-region coverage
# =============================================================================

for table_name in [
    "Efficiency",
    "CostVariable",
    "CostInvest",
    "ETLSegment",
]:

    table = db_encoded[table_name].copy()

    if "region" not in table.columns:
        continue

    region_values = table["region"].dropna().astype(str)

    node_values = region_values[
        ~region_values.str.contains("-", regex=False)
    ]

    edge_values = region_values[
        region_values.str.contains("-", regex=False)
    ]

    invalid_node_regions = sorted(
        set(node_values) - VALID_NODE_REGIONS
    )

    valid_edge_regions = (
        VALID_PIPELINE_EDGE_REGIONS
        | VALID_ROAD_EDGE_REGIONS
    )

    invalid_edge_regions = sorted(
        set(edge_values) - valid_edge_regions
    )

    print(
        f"{table_name}: "
        f"{node_values.nunique():,} node regions, "
        f"{edge_values.nunique():,} edge regions, "
        f"{len(invalid_node_regions):,} invalid nodes, "
        f"{len(invalid_edge_regions):,} invalid edges"
    )

    assert not invalid_node_regions, (
        f"{table_name} has invalid node regions: "
        f"{invalid_node_regions[:10]}"
    )

    assert not invalid_edge_regions, (
        f"{table_name} has invalid edge regions: "
        f"{invalid_edge_regions[:10]}"
    )


pipeline_etl_regions = set(
    db_encoded["ETLSegment"]
    .loc[
        db_encoded["ETLSegment"]["tech_or_group"].isin(PIPE_TECHS),
        "region",
    ]
    .astype(str)
)

pipeline_eff_regions = set(
    db_encoded["Efficiency"]
    .loc[
        db_encoded["Efficiency"]["tech"].isin(PIPE_TECHS),
        "region",
    ]
    .astype(str)
)

assert pipeline_etl_regions == pipeline_eff_regions, (
    "Pipeline ETLSegment region coverage does not match "
    "pipeline Efficiency region coverage."
)

print("\nPipeline ETLSegment coverage matches pipeline Efficiency coverage.")

Efficiency: 2,276 node regions, 8,580 edge regions, 0 invalid nodes, 0 invalid edges
CostVariable: 2,276 node regions, 8,580 edge regions, 0 invalid nodes, 0 invalid edges
CostInvest: 2,276 node regions, 1,896 edge regions, 0 invalid nodes, 0 invalid edges
ETLSegment: 2,276 node regions, 8,580 edge regions, 0 invalid nodes, 0 invalid edges

Pipeline ETLSegment coverage matches pipeline Efficiency coverage.


In [20]:
# =============================================================================
# Clear solver output tables
# =============================================================================

output_tables = [
    table_name
    for table_name in db_encoded
    if table_name.startswith("Output")
]

for table_name in output_tables:
    before_rows = len(db_encoded[table_name])
    db_encoded[table_name] = db_encoded[table_name].iloc[0:0].copy()
    print(f"{table_name}: {before_rows:,} → 0")

OutputCurtailment: 0 → 0
OutputNetCapacity: 1,693 → 0
OutputBuiltCapacity: 1,693 → 0
OutputRetiredCapacity: 0 → 0
OutputFlowIn: 1,608 → 0
OutputFlowOut: 1,608 → 0
OutputStorageLevel: 0 → 0
OutputDualVariable: 0 → 0
OutputObjective: 1 → 0
OutputEmission: 0 → 0
OutputCost: 2,187 → 0


In [21]:
# =============================================================================
# Final encoded database summary
# =============================================================================

final_summary = (
    pd.DataFrame(
        [
            {
                "table": table_name,
                "rows": len(df),
                "columns": len(df.columns),
            }
            for table_name, df in db_encoded.items()
        ]
    )
    .sort_values("table")
    .reset_index(drop=True)
)

display(final_summary)

,table,rows,columns
0,CapacityCredit,0,13
1,CapacityFactorProcess,0,15
2,CapacityFactorTech,0,14
3,CapacityToActivity,0,11
4,Commodity,7,4
...,...,...,...
80,Technology,16,15
81,TechnologyType,4,2
82,TimePeriod,2,3
83,TimePeriodType,2,2


In [22]:
# =============================================================================
# Create fresh SQLite database from raw schema
# =============================================================================

if OUTPUT_SQLITE_PATH.exists():
    OUTPUT_SQLITE_PATH.unlink()

db_mgmt.convert_sql_to_sqlite(
    RAW_SCHEMA_PATH,
    OUTPUT_SQLITE_PATH,
)

print(f"Created fresh SQLite: {OUTPUT_SQLITE_PATH.name}")

Created fresh SQLite: CANOE_geospatial_canada_basemap_1deg_intersects_roads_weak.sqlite


In [23]:
# =============================================================================
# Write encoded tables to fresh SQLite
# =============================================================================

db_mgmt.update_sqlite(
    OUTPUT_SQLITE_PATH,
    db_encoded,
)

print(f"Wrote {len(db_encoded)} tables to {OUTPUT_SQLITE_PATH.name}")

Wrote 85 tables to CANOE_geospatial_canada_basemap_1deg_intersects_roads_weak.sqlite


In [24]:
# =============================================================================
# Verify exported SQLite database
# =============================================================================

db_test = db_mgmt.sqlite_to_dfs(
    OUTPUT_SQLITE_PATH,
)

print(f"Exported tables: {len(db_test)}")

for table_name in [
    "Region",
    "Technology",
    "Demand",
    "LimitCapacity",
    "Efficiency",
    "CostVariable",
    "CostInvest",
    "ETLSegment",
]:
    print(f"{table_name}: {len(db_test[table_name]):,}")

truck_techs = set(truck_tech_specs["tech"])
expected_truck_rows = len(road_links) * len(truck_tech_specs)

assert truck_techs.issubset(set(db_test["Technology"]["tech"]))
assert db_test["Efficiency"]["tech"].isin(truck_techs).sum() == expected_truck_rows
assert db_test["CostVariable"]["tech"].isin(truck_techs).sum() == expected_truck_rows
assert db_test["CostInvest"]["tech"].isin(truck_techs).sum() == expected_truck_rows

print("Exported SQLite validated.")

Exported tables: 85
Region: 2,276
Technology: 16
Demand: 123
LimitCapacity: 4,552
Efficiency: 60,358
CostVariable: 48,732
CostInvest: 12,136
ETLSegment: 155,488
Exported SQLite validated.
